# Pebble Accretion 3 (PA3Py)

This module directly integrates the physical formulations derived in:
- **Ormel (2017)**: *The Emerging Paradigm of Pebble Accretion*.
- **Drążkowska et al. (2023)**: *Planet Formation Theory in the Era of ALMA and Kepler: from Pebbles to Exoplanets*.


### 1. Critical Mass for Pebble Accretion Onset (*Onset Mass*)
A physical barrier appears for very low-mass bodies. To capture the necessary drag gas, a body must have a minimum mass. We implement:
$$ M_{\rm PA\ onset} = {\rm St} \eta^3 M_{\star} $$

*Implementation:* If $M_{\rm pl} < M_{\rm PA\ onset}$, the model does not assume a zero accretion rate, but transitions to the **Planetesimal Accretion (Safronov)** mode (Eq. 7.14 of Ormel 2017), assuming a gas-free ballistic regime with standard rocky density. This provides a realistic transition between early embryo growth and the rapid runaway growth provided by pebbles.


### 2. Dynamical Accretion Equations
The orbital approach regimes are naturally divided according to the fluid properties analyzed by **Ormel (2017)**. 
- **Headwind (Bondi-like drift limit)** ($M \lesssim M_{\rm hw/sh}$):
  $$ \dot{M}_{\rm 2D, hw} = \sqrt{8 G M_{\rm pl} t_{\rm stop} v_{\rm hw}} \Sigma_{\rm peb} $$
- **Shear (Hill limit)** ($M \gtrsim M_{\rm hw/sh}$):
  $$ \dot{M}_{\rm 2D, sh} = 2 R_{\rm Hill}^2 \Omega_{\rm K} {\rm St}^{2/3} \Sigma_{\rm peb} $$
  
With the transition mass evaluated analytically as:
$$ M_{\rm hw/sh} = \frac{v_{\rm hw}^3}{8 G \Omega_{\rm K} t_{\rm stop}} $$


### 3. Analytical Transition to a 3D Disk
We no longer evaluate the error function (`erf`) numerically to force transitions. We use the turbulent approximation for the natural decay factor proposed by **Dubrulle (1995) / Ormel 2017 (Eq 7.24)**:
$$ \dot{M} = \dot{M}_{\rm 2D} \left( \frac{b_{\rm col}}{b_{\rm col} + h_{\rm peb} \sqrt{8/\pi}} \right) $$
Where $h_{\rm peb} = \sqrt{\frac{\alpha_T}{\alpha_T + {\rm St}}} h_{\rm gas}$. This transition converges to the natural 3D limit in highly vertically mixed disks.


### 4. Regulatory Isolation Mass
Numerical barriers and mass overshooting are prevented by the updated gap-opening isolation limit provided by Bitsch, modified under the latest analytical simplification from the comparison in **Drążkowska et al. 2023**:
$$ M_{\rm iso, peb} = 25 M_{\oplus} \left(\frac{H/r}{0.05}\right)^3 \left(\frac{M_{\star}}{M_{\odot}}\right) $$


### 5. Consumption and Drift
Pebbles drift towards the central star, limited by:
$$ v_{\rm r,solid} \approx \frac{- 2\eta v_{\rm K} {\rm St}}{1 + {\rm St}^2} $$
To keep the accretion module purely parasitic and conservative, the dynamical trace from `tripodpy` is extracted assuming direct compatibility, evaluating disk crossings to efficiently share fluxes with the analog approach in the spatial and temporal loop.


### 6. Planetary Composition (Physical Tracking)
Embryos are initialized assuming a `100%` rocky seed (silicates). Composition tracking evaluates the embryo's position relative to the dynamic water *snowline* (`r_snow`) extracted from the base hydrodynamic simulation.
- If $r \ge r_{\rm snow}$: Accreted material assumes an icy composition (50% H2O, 50% Silicates).
- If $r < r_{\rm snow}$: Accreted material assumes a dry dust composition (100% Silicates).

The pipeline evaluates the final taxonomic classification strictly via the water mass fraction: $f_{\rm H2O} = M_{\rm H2O} / (M_{\rm H2O} + M_{\rm sil})$, omitting secondary chemicals such as CO or CO2.


## Limitations and Computational Notes

- **Spatio-Temporal Reservoir**: The code subtracts `flux_consumed` to compute the spatial shadowing of outer planets over inner planets within the same timestep. However, since it is a *post-processing* algorithm based on pre-calculated HDF5 snapshots, it cannot deplete the original dust reservoir of the disk for future timestamps. This is an intrinsic limitation accepted in population synthesis studies due to the partial mitigation provided by the rapid radial drift computed by *TriPoDPy*.
